<a href="https://colab.research.google.com/github/fborrasumh/bioestadistica/blob/main/cuadernos/Tema_190_Simulacion_de_Monte_Carlo_y_distribuciones_muestrales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# <font color='steelblue'>Tema 190. Simulación de Monte Carlo y distribuciones muestrales</font>

**Bioestadística (1964) · Grado en Medicina · 2.º curso · Curso 2027-2028**

**Módulo 4. Probabilidad**

**Autor:** Fernando Borrás Rocher · Universidad Miguel Hernández de Elche · ORCID [0000-0002-5519-4573](https://orcid.org/0000-0002-5519-4573)

**Duración:** 50 minutos de clase, más el estudio de los apartados de ampliación

**Práctica:** [unidad 190 de la app Bioestadística](https://fborrasumh.github.io/bioestadistica/)

**Evaluación:** el examen del Módulo 4 se hace **a mano**. Resuelve primero cada ejercicio con papel y calculadora, y usa el código solo para comprobar.

**Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es). Todos los pacientes y datos de este cuaderno son sintéticos.

*Antes de trabajar, guarda tu copia: **Archivo → Guardar una copia en Drive**.*

## <font color='steelblue'>Configuración del cuaderno</font>

Ejecuta esta celda al empezar. Instala el módulo de preguntas interactivas y prepara el cuaderno.

In [ ]:
#@title Ejecutar configuración
%pip install -q jupyterquiz==2.9.6.4
from jupyterquiz import display_quiz
print("Cuaderno configurado")

## <font color='steelblue'>Objetivos de aprendizaje</font>

Al terminar esta sesión serás capaz de:

1. Explicar en qué consiste el método de Monte Carlo y para qué sirve.
2. Distinguir la distribución de una variable en la población de la **distribución muestral** de un estadístico.
3. Calcular e interpretar el **error estándar** de una media y de una proporción.
4. Enunciar el **teorema central del límite** y comprobarlo por simulación con datos asimétricos.
5. Explicar cómo influye el tamaño de la muestra en la precisión de una estimación.
6. Medir la variabilidad de las respuestas de un modelo de lenguaje y reconocer la sobredispersión.

## <font color='steelblue'>Plan de la sesión</font>

| Bloque | Minutos |
| --- | --- |
| 1. Caso clínico de partida | 5 |
| 2. Conceptos clave | 10 |
| 3. Demostración: muestras simuladas | 20 |
| 4. Práctica en la app Bioestadística | 10 |
| 5. Autoevaluación | 5 |

## <font color='steelblue'>1. Caso clínico de partida (5 min)</font>

Dos residentes estudian la **estancia hospitalaria** de los pacientes con neumonía de su hospital. Cada uno revisa al azar 30 historias del último año. El primero obtiene una estancia media de 6,2 días; el segundo, de 7,9 días.

¿Se ha equivocado alguno? No necesariamente. **Cada muestra es distinta**, y por tanto también lo es su media. Si un tercer residente revisara otras 30 historias, obtendría un tercer valor.

Esta variabilidad entre muestras no es un defecto: es inevitable, y se puede **cuantificar**. Saber cuánto varía una media de una muestra a otra es la base de los intervalos de confianza y de los contrastes de hipótesis del Módulo 5. Hoy lo veremos simulando miles de muestras con el ordenador: el método de **Monte Carlo**.

## <font color='steelblue'>2. Conceptos clave (10 min)</font>

### <font color='steelblue'>2.1 El método de Monte Carlo</font>

Consiste en estimar una cantidad **repitiendo muchas veces un experimento aleatorio simulado** y resumiendo los resultados. Ya lo hemos usado para comprobar cálculos de probabilidad (Temas 140 a 170). Hoy lo usaremos para algo que sería imposible en la realidad: extraer miles de muestras de la misma población.

Se llama así por el casino de Montecarlo. Lo desarrollaron en los años cuarenta Stanislaw Ulam y John von Neumann.

### <font color='steelblue'>2.2 Población, muestra y distribución muestral</font>

| Distribución | Qué describe | Su dispersión |
| --- | --- | --- |
| De la variable en la **población** | Cómo varían los **pacientes** | Desviación típica σ |
| De la variable en **una muestra** | Cómo varían los pacientes de esa muestra | Desviación típica muestral *s* |
| **Muestral** de la media | Cómo varía la **media** de una muestra a otra | **Error estándar** |

La **distribución muestral** de un estadístico es la distribución de los valores que tomaría ese estadístico en todas las muestras posibles del mismo tamaño.

### <font color='steelblue'>2.3 El error estándar</font>

| Estadístico | Error estándar |
| --- | --- |
| Media | $EE(\bar{x}) = \dfrac{\sigma}{\sqrt{n}} \approx \dfrac{s}{\sqrt{n}}$ |
| Proporción | $EE(\hat{p}) = \sqrt{\dfrac{p(1-p)}{n}}$ |

El error estándar disminuye con la raíz cuadrada de *n*: para reducirlo a la mitad hay que **cuadruplicar** la muestra.

**No confundas** la desviación típica (dispersión de los pacientes) con el error estándar (precisión de la media). Al aumentar la muestra, la desviación típica de los pacientes no cambia; el error estándar, sí.

### <font color='steelblue'>2.4 El teorema central del límite</font>

Si se toman muestras de tamaño *n* suficientemente grande de **cualquier** población con media μ y desviación típica σ, la distribución de la media muestral se aproxima a una normal:

$$\bar{x} \approx N\left(\mu, \frac{\sigma}{\sqrt{n}}\right)$$

aunque la variable original **no** sea normal. Con distribuciones moderadamente asimétricas suele bastar con *n* ≥ 30; con distribuciones muy asimétricas hace falta más. Este teorema explica por qué la normal aparece en casi toda la inferencia estadística.

## <font color='steelblue'>3. Demostración: muestras simuladas (20 min)</font>

### <font color='steelblue'>3.1 Una población sintética de pacientes</font>

Creamos una población de 100.000 ingresos por neumonía con una estancia **asimétrica**: la mayoría se va en pocos días y unos pocos permanecen mucho tiempo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=190)
poblacion = rng.lognormal(mean=np.log(5.5), sigma=0.55, size=100_000).round(1) + 1

mu, sigma = poblacion.mean(), poblacion.std()
print(f"Media poblacional μ = {mu:.2f} días · desviación típica σ = {sigma:.2f} días")

plt.figure(figsize=(7, 4))
plt.hist(poblacion, bins=80, color="lightsteelblue", edgecolor="white")
plt.axvline(mu, color="firebrick", label=f"μ = {mu:.1f} días")
plt.xlim(0, 30)
plt.xlabel("Estancia (días)")
plt.ylabel("Número de pacientes")
plt.title("Población: estancia de 100.000 ingresos por neumonía")
plt.legend()
plt.show()

### <font color='steelblue'>3.2 Una muestra, dos muestras, diez muestras</font>

In [ ]:
for i in range(1, 11):
    muestra = rng.choice(poblacion, size=30, replace=False)
    print(f"Muestra {i:>2}: media {muestra.mean():.2f} días")

Cada residente obtendría un valor distinto, todos alrededor de μ. ¿Cuánto varían? Lo vemos con 10.000 muestras.

### <font color='steelblue'>3.3 La distribución muestral de la media</font>

In [ ]:
def medias_muestrales(n, repeticiones=10_000):
    """Medias de muchas muestras aleatorias de tamaño n de la población."""
    return np.array([rng.choice(poblacion, size=n).mean() for _ in range(repeticiones)])


fig, ejes = plt.subplots(1, 3, figsize=(13, 3.8), sharex=True)
for eje, n in zip(ejes, [5, 30, 120]):
    medias = medias_muestrales(n)
    ee_teorico = sigma / np.sqrt(n)
    eje.hist(medias, bins=60, density=True, color="lightsteelblue", edgecolor="white")
    x = np.linspace(medias.min(), medias.max(), 200)
    eje.plot(x, np.exp(-((x - mu) ** 2) / (2 * ee_teorico ** 2)) / (ee_teorico * np.sqrt(2 * np.pi)),
             color="firebrick", label="Normal del TCL")
    eje.set_title(f"n = {n}\nEE simulado {medias.std():.2f} · teórico {ee_teorico:.2f}")
    eje.set_xlabel("Media de la muestra (días)")
ejes[0].legend()
plt.tight_layout()
plt.show()

Tres observaciones fundamentales:

1. Las medias se concentran **alrededor de μ**: la media muestral no tiene sesgo.
2. Al aumentar *n*, la distribución se **estrecha**: el error estándar simulado coincide con σ/√*n*.
3. Aunque la población es muy asimétrica, con *n* = 30 o más la distribución de la media ya es **casi normal**: es el teorema central del límite. Con *n* = 5 todavía se aprecia la asimetría.

### <font color='steelblue'>3.4 El 95 % de las medias</font>

Si las medias siguen aproximadamente una normal, el 95 % de ellas estará a menos de 1,96 errores estándar de μ. Lo comprobamos:

In [ ]:
n = 30
medias = medias_muestrales(n)
ee = sigma / np.sqrt(n)
dentro = np.abs(medias - mu) < 1.96 * ee
print(f"Medias a menos de 1,96 EE de μ: {dentro.mean():.1%}")

Esta es la idea del **intervalo de confianza** del Tema 200: si cada residente construye el intervalo media ± 1,96 EE con su muestra, alrededor del 95 % de esos intervalos contendrá el verdadero valor μ.

### <font color='steelblue'>3.5 Tamaño muestral y precisión</font>

In [ ]:
tamanos = np.array([10, 20, 40, 80, 160, 320, 640])
ee_simulado = [medias_muestrales(n, 3000).std() for n in tamanos]

plt.figure(figsize=(7, 4))
plt.plot(tamanos, ee_simulado, marker="o", label="Simulado")
plt.plot(tamanos, sigma / np.sqrt(tamanos), linestyle="--", label="σ / √n")
plt.xscale("log")
plt.xticks(tamanos, tamanos)
plt.xlabel("Tamaño de la muestra (escala logarítmica)")
plt.ylabel("Error estándar de la media (días)")
plt.title("Cuadruplicar la muestra reduce el error estándar a la mitad")
plt.legend()
plt.show()

### <font color='steelblue'>3.6 La distribución muestral de una proporción</font>

Lo mismo ocurre con las proporciones. Si la prevalencia real de diabetes en una población es del 18 %, ¿cuánto varía la prevalencia estimada en muestras de 100 pacientes?

**A mano:** $EE = \sqrt{0{,}18 \times 0{,}82 / 100} \approx 0{,}038$. El 95 % de las muestras dará una prevalencia entre 0,18 ± 1,96 × 0,038, es decir, entre el 10,5 % y el 25,5 % aproximadamente.

In [ ]:
p = 0.18
prevalencias = rng.binomial(n=100, p=p, size=10_000) / 100
print(f"EE simulado: {prevalencias.std():.3f} · teórico: {np.sqrt(p * (1 - p) / 100):.3f}")
print(f"Percentiles 2,5 y 97,5 de la prevalencia estimada: {np.percentile(prevalencias, [2.5, 97.5])}")

### <font color='steelblue'>3.7 La variabilidad de un modelo de lenguaje</font>

En el Tema 170 modelamos el número de aciertos de un modelo de IA con una binomial. Ese modelo asume que **todas las preguntas tienen la misma probabilidad de acierto**. En la práctica, unas preguntas son fáciles para el modelo y otras muy difíciles. Simulamos 50 preguntas con dificultad variable y repetimos el «examen» muchas veces.

In [ ]:
n_preguntas, repeticiones = 50, 5000

# Probabilidad de acierto de cada pregunta: media 0,8, pero muy desigual
p_pregunta = rng.beta(a=2, b=0.5, size=n_preguntas)
print(f"Probabilidad media de acierto: {p_pregunta.mean():.2f}")

# Cada repetición del examen responde a las mismas 50 preguntas
aciertos_reales = (rng.random((repeticiones, n_preguntas)) < p_pregunta).sum(axis=1)
# Lo que predeciría una binomial con la misma media
aciertos_binomial = rng.binomial(n_preguntas, p_pregunta.mean(), size=repeticiones)

print(f"Varianza con dificultad desigual: {aciertos_reales.var():.2f}")
print(f"Varianza de la binomial:          {aciertos_binomial.var():.2f}")

Aquí ocurre algo contraintuitivo: **repitiendo el mismo examen**, la variabilidad es *menor* que la de la binomial, porque las preguntas fáciles se aciertan casi siempre y las difíciles casi nunca. En cambio, si cada repetición es un examen de un **área distinta** (unas áreas le resultan fáciles al modelo y otras difíciles), la probabilidad media de acierto cambia de un examen a otro y la variabilidad es *mayor* que la binomial: es la **sobredispersión**.

In [ ]:
p_area = rng.beta(a=8, b=2, size=repeticiones)          # acierto medio de cada área: 0,8 en promedio
examenes_por_area = rng.binomial(n_preguntas, p_area)
p_media = p_area.mean()
print(f"Varianza con exámenes de áreas distintas: {examenes_por_area.var():.2f}")
print(f"Varianza de la binomial con p = {p_media:.2f}:    {n_preguntas * p_media * (1 - p_media):.2f}")

**Conclusión práctica:** antes de aplicar un modelo de probabilidad a los resultados de un modelo de IA, hay que preguntarse **qué** se repite (la misma pregunta, el mismo examen o exámenes distintos) y si la probabilidad de acierto es realmente constante.

### <font color='steelblue'>Ampliación: la misma pregunta, muchas veces, al modelo real</font>

Con una clave configurada, esta celda repite 10 veces una misma pregunta con temperatura alta y cuenta las respuestas distintas. Sin clave, no hace nada.

Esta celda prepara la conexión con el modelo de lenguaje (Tema 80). Lee la clave de Secretos sin mostrarla. Si no hay clave, el cuaderno funciona con respuestas de ejemplo. Si el modelo indicado ya no existe, cambia la variable `MODELO`.

In [ ]:
%pip install -q google-genai

MODELO = "gemini-2.5-flash"

try:
    from google.colab import userdata
    from google import genai
    from google.genai import types
    cliente = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
    print("Conexión preparada con el modelo", MODELO)
except Exception as error:
    cliente = None
    print("Sin clave o fuera de Colab: se usarán respuestas de ejemplo.", type(error).__name__)


def preguntar(prompt, temperatura=0.2):
    """Envía un prompt al modelo y devuelve su texto (None si no hay conexión)."""
    if cliente is None:
        return None
    respuesta = cliente.models.generate_content(
        model=MODELO,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=temperatura),
    )
    return respuesta.text

In [ ]:
from collections import Counter

pregunta = ("Un paciente con fiebre, tos productiva y crepitantes en la base derecha. "
            "¿Cuál es el diagnóstico más probable? Responde con dos palabras como máximo.")

if cliente is not None:
    respuestas = [preguntar(pregunta, temperatura=1.5).strip().lower().rstrip(".") for _ in range(10)]
    print(Counter(respuestas))
else:
    print("Configura tu clave para hacer el experimento con el modelo real.")

### <font color='steelblue'>Ampliación: errores frecuentes</font>

**Confundir desviación típica y error estándar** al leer un artículo: una media «± 0,4» puede ser una desviación típica muy pequeña o un error estándar con una dispersión grande.

**Pensar que la muestra debe ser una fracción grande de la población:** la precisión depende sobre todo del tamaño absoluto de la muestra, no del porcentaje de la población que representa.

**Aplicar el teorema central del límite a muestras muy pequeñas** de variables muy asimétricas.

**Creer que el TCL hace normales los datos:** lo que se aproxima a la normal es la **media** de muchas muestras, no la distribución de los pacientes.

**Olvidar que la simulación tiene su propio error:** con pocas repeticiones, el resultado de un Monte Carlo también varía.

## <font color='steelblue'>4. Práctica en la app Bioestadística (10 min)</font>

Abre la **[unidad 190 de la app Bioestadística](https://fborrasumh.github.io/bioestadistica/)** y completa sus misiones:

1. **Misión básica (sin asistente):** calcular a mano el error estándar de medias y proporciones con distintos tamaños muestrales.
2. **Misión clínica (con pistas):** simular la distribución muestral de la prevalencia de hipertensión con muestras de 50, 200 y 800 pacientes.
3. **Misión reto (con asistente):** repetir 20 veces una pregunta clínica a un modelo de IA, registrar las respuestas y discutir si un modelo binomial describe su variabilidad.

Al terminar, exporta el **informe de la unidad**.

## <font color='steelblue'>5. Autoevaluación (5 min)</font>

Responde a mano, sin asistente y sin ejecutar código.

In [ ]:
#@title Pregunta interactiva
preguntas = [{'question': 'La PAS de una población tiene σ = 16 mmHg. ¿Cuál es el error estándar de la media '
              'en muestras de 64 pacientes?',
  'type': 'multiple_choice',
  'answers': [{'answer': '2 mmHg', 'correct': True, 'feedback': 'Correcto: 16 / √64 = 16 / 8.'},
              {'answer': '16 mmHg',
               'correct': False,
               'feedback': 'Esa es la desviación típica de los pacientes.'},
              {'answer': '0,25 mmHg',
               'correct': False,
               'feedback': 'Eso es 16 / 64: hay que dividir entre la raíz de n.'},
              {'answer': '4 mmHg',
               'correct': False,
               'feedback': 'Revisa la raíz cuadrada de 64.'}]},
 {'question': 'Para reducir a la mitad el error estándar de una media, ¿cuánto hay que aumentar la '
              'muestra?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Multiplicarla por 4',
               'correct': True,
               'feedback': 'Correcto: el EE depende de 1/√n, y √4 = 2.'},
              {'answer': 'Multiplicarla por 2',
               'correct': False,
               'feedback': 'Con el doble de muestra, el EE solo se divide entre √2 ≈ 1,41.'},
              {'answer': 'Sumarle 30 pacientes',
               'correct': False,
               'feedback': 'El efecto depende de la proporción, no de una cantidad fija.'},
              {'answer': 'No se puede reducir',
               'correct': False,
               'feedback': 'Aumentar la muestra siempre lo reduce.'}]},
 {'question': '¿Qué afirma el teorema central del límite?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Que la media muestral se aproxima a una normal con muestras '
                         'suficientemente grandes, sea cual sea la distribución de la variable',
               'correct': True,
               'feedback': 'Correcto.'},
              {'answer': 'Que toda variable biológica sigue una distribución normal',
               'correct': False,
               'feedback': 'Muchas variables no son normales.'},
              {'answer': 'Que con muestras grandes los datos de los pacientes se vuelven normales',
               'correct': False,
               'feedback': 'Los datos no cambian; es la media la que se aproxima a la normal.'},
              {'answer': 'Que la media muestral es siempre igual a la poblacional',
               'correct': False,
               'feedback': 'La media muestral varía alrededor de la poblacional.'}]},
 {'question': 'Una prevalencia real del 50 % se estima con muestras de 100 personas. ¿Cuál es el '
              'error estándar de la proporción?',
  'type': 'multiple_choice',
  'answers': [{'answer': '0,05',
               'correct': True,
               'feedback': 'Correcto: √(0,5 × 0,5 / 100) = √0,0025.'},
              {'answer': '0,5',
               'correct': False,
               'feedback': 'Esa es la prevalencia, no su error estándar.'},
              {'answer': '0,25',
               'correct': False,
               'feedback': 'Eso es p(1 − p); falta dividir entre n y hacer la raíz.'},
              {'answer': '0,0025',
               'correct': False,
               'feedback': 'Esa es la varianza; falta la raíz cuadrada.'}]},
 {'question': 'Al pasar de 50 a 500 pacientes en un estudio, ¿qué ocurre con la desviación típica '
              'de la edad de los pacientes?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Se mantiene aproximadamente igual',
               'correct': True,
               'feedback': 'Correcto: la dispersión de los pacientes no depende del tamaño de la '
                           'muestra.'},
              {'answer': 'Disminuye a un tercio',
               'correct': False,
               'feedback': 'Eso le ocurre, aproximadamente, al error estándar (√10 ≈ 3,2).'},
              {'answer': 'Aumenta',
               'correct': False,
               'feedback': 'No hay motivo para que aumente.'},
              {'answer': 'Se hace cero',
               'correct': False,
               'feedback': 'Los pacientes siguen teniendo edades distintas.'}]}]
display_quiz(preguntas)

## <font color='steelblue'>Ideas clave</font>

- El método de Monte Carlo estima cantidades repitiendo muchas veces un experimento simulado.
- La media de una muestra varía de una muestra a otra; su dispersión es el error estándar, σ/√*n*.
- El teorema central del límite hace que la media muestral sea aproximadamente normal con muestras grandes.
- Alrededor del 95 % de las medias muestrales está a menos de 1,96 EE de la media poblacional: es la base del intervalo de confianza.
- Antes de modelar los aciertos de un modelo de IA, hay que comprobar si la probabilidad de acierto es constante.

## <font color='steelblue'>Glosario</font>

| Término | Definición |
| --- | --- |
| Método de Monte Carlo | Estimación de una cantidad mediante la repetición de experimentos aleatorios simulados |
| Estadístico | Valor calculado a partir de una muestra |
| Distribución muestral | Distribución de un estadístico en todas las muestras posibles de un mismo tamaño |
| Error estándar | Desviación típica de la distribución muestral de un estadístico |
| Teorema central del límite | Resultado según el cual la media muestral se aproxima a una normal al aumentar *n* |
| Estimador insesgado | Estimador cuya distribución muestral está centrada en el parámetro |
| Sobredispersión | Variabilidad mayor que la que predice el modelo de probabilidad supuesto |

## <font color='steelblue'>Para saber más</font>

- Altman DG, Bland JM. Standard deviations and standard errors. *BMJ*. 2005;331(7521):903.
- Metropolis N, Ulam S. The Monte Carlo method. *J Am Stat Assoc*. 1949;44(247):335-341.

---
*Tema 190 · Bioestadística (1964) · Grado en Medicina · Universidad Miguel Hernández de Elche · Fernando Borrás Rocher · CC BY 4.0*